In [1]:
from moabb.paradigms import FilterBankMotorImagery
from moabb.datasets import *
from sklearn.metrics import get_scorer

dataset=AlexMI()

paradigm = FilterBankMotorImagery(n_classes=len(dataset.event_id),resample=250)    

cache_config = dict(
    use=True,
    save_raw=False,
    save_epochs=False,
    save_array=True,
    overwrite_raw=False,
    overwrite_epochs=False,
    overwrite_array=False,
)
scorer = get_scorer(paradigm.scoring)


Choosing from all possible events


In [12]:
from sklearn.model_selection import StratifiedKFold

n_blocks=4
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
n_blocks_grid = list(range(1,n_blocks+1))
#theta_grid = [0,0.1,0.2,0.3,0.4,0.5,0.6, 0.7, 0.8, 0.9,1]
theta_grid=[0,0.2,1]

In [13]:
job_args = []
for subject in dataset.subject_list[:1]:
    X, y, meta = paradigm.get_data(dataset=dataset, subjects=[subject], cache_config=cache_config,)
    for fold, (train_idc, test_idc) in enumerate(cv.split(X, y)):
        for theta in theta_grid:
            job_args.append((dataset.code, subject, fold, X,y, train_idc, test_idc, theta))

In [14]:
from hoda.hoda import BTTDA
from hoda.classification import ZScore, ZLogRatio
from hoda.tensorize import fh_power, fh_log_envelope
from sklearn.preprocessing import StandardScaler
from classification_mi import make_clf, get_hoda_params, get_bttda_params
import tensorly as tl
import pandas as pd
import warnings

def eval_fold(dataset, subject, fold, X, y, train_idc, test_idc, theta):
    X = fh_log_envelope(X, sfreq=250, target_sfreq=16)
    X_st=X
    #X_st = ZLogRatio().fit(X[train_idc], y[train_idc]).transform(X,y)
    X_st = ZScore().fit(X_st[train_idc], y[train_idc]).transform(X_st,y) 
    
    hoda_params = get_hoda_params()
    hoda_params['theta'] = theta
    bttda = BTTDA(
        ranks=[None]*max(n_blocks_grid),
        hoda_params=hoda_params,
        verbose=False,        
    )
    
    bttda.fit(X_st[train_idc], y[train_idc])
    print(bttda.n_blocks_)
    clf = make_clf()
    result = []
    for n_blocks in n_blocks_grid:
        if n_blocks > bttda.n_blocks_:
            break
        Xt = bttda.transform(X_st, n_blocks=n_blocks)
        X_rec = bttda.inv_transform(Xt, n_blocks=n_blocks)
        try:
            clf.fit(Xt[train_idc], y[train_idc])
        except ValueError as e:
            warnings.warn(str(e))
            break
        result.append(dict(
            subject = subject,
            dataset = dataset,
            fold = fold,
            theta=theta,
            n_blocks=n_blocks,
            train_score = scorer(clf, Xt[train_idc], y[train_idc]),
            test_score = scorer(clf, Xt[test_idc], y[test_idc]),
            train_mse = tl.metrics.regression.MSE(X_st[train_idc], X_rec[train_idc]),
            test_mse = tl.metrics.regression.MSE(X_st[test_idc], X_rec[test_idc]),          
        ))
    return pd.DataFrame(result)

        

In [15]:
import joblib
from joblib import Parallel, delayed
from hpc import create_cluster, create_client, TIMEOUT

with create_cluster(cluster='local') as cluster, create_client(cluster) as client:
    with joblib.parallel_backend('dask', wait_for_workers_timeout=TIMEOUT): 
        results = Parallel(n_jobs=len(job_args), verbose=True)(delayed(eval_fold)(*args) for args in job_args)
results = pd.concat(results, ignore_index=True)

/vsc-hard-mounts/leuven-data/352/vsc35289/hoda-bci/src/hoda/hoda.py:360: UserWarning:

Maximum number of iterations reached without convergence



4


/vsc-hard-mounts/leuven-data/352/vsc35289/hoda-bci/src/hoda/hoda.py:360: UserWarning:

Maximum number of iterations reached without convergence



4
4


/vsc-hard-mounts/leuven-data/352/vsc35289/hoda-bci/src/hoda/hoda.py:360: UserWarning:

Maximum number of iterations reached without convergence



...

/vsc-hard-mounts/leuven-data/352/vsc35289/hoda-bci/src/hoda/hoda.py:360: UserWarning:

Maximum number of iterations reached without convergence



4
4
4


/vsc-hard-mounts/leuven-data/352/vsc35289/hoda-bci/src/hoda/hoda.py:360: UserWarning:

Maximum number of iterations reached without convergence



4
4


[Parallel(n_jobs=15)]: Done  15 out of  15 | elapsed:   38.8s finished


In [16]:
import plotly.io as pio
pio.renderers.default = 'iframe'

In [17]:
results.to_csv('results/gridsearch_mi.csv')

In [18]:
results

,subject,dataset,fold,theta,n_blocks,train_score,test_score,train_mse,test_mse
0,1,AlexandreMotorImagery,0,0.0,1,0.958333,0.250000,9.953218e-01,1.008738e+00
1,1,AlexandreMotorImagery,0,0.0,2,1.000000,0.500000,9.925037e-01,1.008759e+00
2,1,AlexandreMotorImagery,0,0.0,3,1.000000,0.416667,9.888962e-01,1.009307e+00
3,1,AlexandreMotorImagery,0,0.0,4,1.000000,0.583333,9.861737e-01,1.017506e+00
4,1,AlexandreMotorImagery,0,0.2,1,1.000000,0.416667,9.859789e-01,2.225245e+00
5,1,AlexandreMotorImagery,0,0.2,2,1.000000,0.416667,9.789817e-01,2.224365e+00
6,1,AlexandreMotorImagery,0,0.2,3,1.000000,0.416667,9.725302e-01,2.232688e+00
7,1,AlexandreMotorImagery,0,0.2,4,1.000000,0.416667,9.666513e-01,2.258412e+00
8,1,AlexandreMotorImagery,0,1.0,1,0.583333,0.416667,5.641233e-31,6.123596e-31
9,1,AlexandreMotorImagery,0,1.0,2,0.583333,0.416667,0.000000e+00,0.000000e+00


In [19]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'iframe'

df = results.groupby(['n_blocks', 'theta'])['test_score'].aggregate('mean').reset_index()
n_colors = df['theta'].nunique()
colors = px.colors.sample_colorscale("viridis", [n/(n_colors -1) for n in range(n_colors)])
fig = px.line(df, x='n_blocks', y='test_score', color='theta', color_discrete_sequence=colors)
fig.show()

In [20]:
import seaborn as sns
import matplotlib.pyplot as plt

df = results.groupby(['n_blocks', 'theta'])['test_mse'].aggregate('mean').reset_index()
n_colors = df['theta'].nunique()
colors = px.colors.sample_colorscale("viridis", [n/(n_colors -1) for n in range(n_colors)])
fig = px.line(df, x='n_blocks', y='test_mse', color='theta', color_discrete_sequence=colors)
fig.show()

In [22]:
import seaborn as sns
import matplotlib.pyplot as plt

df = results.groupby(['n_blocks', 'theta'])['train_mse'].aggregate('mean').reset_index()
n_colors = df['theta'].nunique()
colors = px.colors.sample_colorscale("viridis", [n/(n_colors -1) for n in range(n_colors)])
fig = px.line(df, x='n_blocks', y='train_mse', color='theta', color_discrete_sequence=colors)
fig.show()